In [1]:
import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomTranslation
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

2025-11-28 14:56:30.245586: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764341790.475440      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764341790.544428      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 8

train_data = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/final-assigment-sp-gmat/final-assignment-gmat/train',
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_split=0.2,
    subset='training',
    seed=42
)

val_data = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/final-assigment-sp-gmat/final-assignment-gmat/train',
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    validation_split=0.2,
    subset='validation',
    seed=42
)

test_data = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/final-assigment-sp-gmat/final-assignment-gmat/test',
    label_mode=None,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

filenames = test_data.file_paths
filenames = [os.path.basename(f) for f in filenames]

test_data = test_data.map(lambda x: preprocess_input(x))

Found 315 files belonging to 2 classes.
Using 252 files for training.


I0000 00:00:1764341806.768668      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1764341806.769253      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 315 files belonging to 2 classes.
Using 63 files for validation.
Found 215 files.


In [3]:
from tensorflow.keras.layers import RandomContrast, RandomBrightness
data_augmentation = Sequential([
    RandomFlip('horizontal'),
    RandomFlip('vertical'),
    RandomRotation(0.3),
    RandomZoom(0.3), 
    RandomTranslation(0.3, 0.3), 
    RandomContrast(0.3),
])

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    mode='min'
)

In [4]:
base_model = EfficientNetV2B0(
    weights='imagenet', 
    input_shape=(*IMAGE_SIZE, 3), 
    include_top=False
)
base_model.trainable = False

model = Sequential([
    data_augmentation,
    tf.keras.layers.Lambda(lambda x: preprocess_input(x)),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.4),
    Dense(2, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
history1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30,
    shuffle=True,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/30


E0000 00:00:1764341822.202896      47 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/sequential_1_1/efficientnetv2-b0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1764341824.917925     115 cuda_dnn.cc:529] Loaded cuDNN version 90300


32/32 ━━━━━━━━━━━━━━━━━━━━ 20s 173ms/step - accuracy: 0.5827 - loss: 0.7207 - val_accuracy: 0.8254 - val_loss: 0.3783 - learning_rate: 0.0010
Epoch 2/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.7701 - loss: 0.4318 - val_accuracy: 0.9048 - val_loss: 0.2814 - learning_rate: 0.0010
Epoch 3/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.8404 - loss: 0.3144 - val_accuracy: 0.9524 - val_loss: 0.1813 - learning_rate: 0.0010
Epoch 4/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.8892 - loss: 0.2827 - val_accuracy: 0.9683 - val_loss: 0.1181 - learning_rate: 0.0010
Epoch 5/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9037 - loss: 0.2511 - val_accuracy: 0.8889 - val_loss: 0.2262 - learning_rate: 0.0010
Epoch 6/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8901 - loss: 0.2918 - val_accuracy: 0.9683 - val_loss: 0.1204 - learning_rate: 0.0010
Epoch 7/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9408 - loss: 0.1619 - val_accuracy: 0.

In [6]:
early_stop2 = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
)

reduce_lr2 = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    mode='min'
)

base_model.trainable = True

#for layer in base_model.layers[:-30]:
#    layer.trainable=False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [7]:
history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    shuffle=True,
    callbacks=[early_stop2, reduce_lr2]
)

Epoch 1/25


E0000 00:00:1764341886.923734      47 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/sequential_1_1/efficientnetv2-b0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


32/32 ━━━━━━━━━━━━━━━━━━━━ 51s 217ms/step - accuracy: 0.7794 - loss: 0.5158 - val_accuracy: 0.9206 - val_loss: 0.2257 - learning_rate: 1.0000e-04
Epoch 2/25
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 110ms/step - accuracy: 0.8864 - loss: 0.2953 - val_accuracy: 0.9365 - val_loss: 0.2024 - learning_rate: 1.0000e-04
Epoch 3/25
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 107ms/step - accuracy: 0.9565 - loss: 0.1777 - val_accuracy: 0.9365 - val_loss: 0.2005 - learning_rate: 1.0000e-04
Epoch 4/25
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 107ms/step - accuracy: 0.9295 - loss: 0.1979 - val_accuracy: 0.9365 - val_loss: 0.1735 - learning_rate: 1.0000e-04
Epoch 5/25
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 110ms/step - accuracy: 0.9380 - loss: 0.1826 - val_accuracy: 0.9683 - val_loss: 0.0930 - learning_rate: 1.0000e-04
Epoch 6/25
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 107ms/step - accuracy: 0.9415 - loss: 0.1534 - val_accuracy: 0.9365 - val_loss: 0.2036 - learning_rate: 1.0000e-04
Epoch 7/25
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 109ms/step - accuracy: 0.9416 - l

In [8]:
y_pred = model.predict(test_data)
y_pred_classes = y_pred.argmax(axis=1)

27/27 ━━━━━━━━━━━━━━━━━━━━ 5s 104ms/step


In [9]:
class_names = train_data.class_names
print(class_names)

['doraemon', 'sonic']


In [10]:
labels = [class_names[idx] for idx in y_pred_classes]

df = pd.DataFrame({
    'img_name': filenames, #from (defined) in code block 2
    'label': labels
})

print(df['label'].value_counts())

label
sonic       124
doraemon     91
Name: count, dtype: int64


In [11]:
df.to_csv("DL.csv", index=False)